# CSTR Dilution-Rate Sweep — Running Usecase 03 as a Chemostat

**The situation:** [usecase 03](03_grow_ecoli_on_acetic_acid.ipynb) grew
*E. coli* on acetic acid in a **batch** bottle — inoculate once, watch
substrate deplete and biomass grow until it's done. Run the same organism
and substrate instead as a **CSTR (continuous stirred-tank reactor)**: fresh
sterile medium flows in at a constant volumetric rate $Q$, and broth flows
out at that *same* rate, so the working liquid volume $V$ never changes.
That equal-in/equal-out condition is what makes this a **chemostat** — the
one parameter that fully characterises its operating point is the
**dilution rate** $D = Q/V$ (units 1/h, the reciprocal of the mean
residence time $\tau = 1/D$).

Left running long enough at a fixed $D$, a chemostat settles onto a
**steady state**: biomass growth is exactly balanced by washout ($\mu = D$
at steady state), and substrate is exactly balanced by the difference
between what arrives in the feed and what the culture consumes.

**Aim of this notebook.** Build one mechanistic CSTR model — the same
`ControlVolume`/`Simulation` machinery usecase 03 used for a batch bottle,
now with a `LiquidFeed`/`LiquidDrain` pair added — and run *it alone* to
steady state across a sweep of dilution rates. Because that one model
already tracks O₂ transfer and the acid-base speciation directly, a single
mechanistic run gives every quantity of interest at once: steady-state
substrate, biomass, dissolved O₂, and pH, plus the volumetric productivity
they imply — no separate oxygen- or pH-specific model is needed to get at
any of them. A closed-form **substrate-only** estimate (Section 3) is kept
alongside purely as a sanity check and to pick a sensible sweep range; it
assumes O₂ is never limiting, and Section 6's own DO panel shows that
assumption holds closely only for part of the range this notebook sweeps.

This is the extension [usecase 03's own "where to go
next"](03_grow_ecoli_on_acetic_acid.ipynb) section pointed to, and the same
setup `demos/builder/cstr_fermenter.py` demonstrates as a plain script;
this notebook adds the dilution-rate sweep and the single publication-style
summary figure that script doesn't produce.

In [ ]:
import sys
from pathlib import Path
import math
import warnings
import numpy as np
import matplotlib.pyplot as plt

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

sys.path.insert(0, str(_find_repo() / "models"))

from PyOMES.chemistry.species import Species
from PyOMES.chemistry.common_species import (
    H2O, H_plus, OH_minus,
    H3PO4, H2PO4_minus, HPO4_2minus, PO4_3minus,
    NH3, NH4_plus,
    CO2, HCO3_minus, CO3_2minus,
)
from PyOMES.chemistry import HenryEquilibrium
from PyOMES.reactions import (
    EquilibriumReaction, ReactionBuilder, ReactionSystem, StoichiometryEntry,
)
from PyOMES.core import (
    ControlVolume, GasPhase, LiquidPhase,
    KineticTransferModel, EquilibriumTransferModel,
    Simulation, GasFeed, PressureReliefVent, LiquidFeed, LiquidDrain,
)
from PyOMES.core.phases import R_L_ATM_MOL_K

# Same rationale as usecase 03: the explicit-Euler CV solver clamps a
# species' removal rate when a step would otherwise drive it negative
# (AccuracyWarning), and separately flags element/charge drift above
# tolerance (ConservationWarning). Silenced only so the run cells below
# stay readable -- Section 3 checks the actual steady state directly
# against the analytical chemostat solution.
from PyOMES.monitoring.accuracy import AccuracyWarning
from PyOMES.monitoring.conservation import ConservationWarning
warnings.filterwarnings("ignore", category=AccuracyWarning)
warnings.filterwarnings("ignore", category=ConservationWarning)

def _e(sp, coeff, phase="liquid"):
    return StoichiometryEntry(species=sp, phase=phase, coefficient=coeff)

print("Imports OK")


## 1  Chemistry and organism — identical to usecase 03

Same eight equilibrium reactions (water autoionization, the three-step
phosphate ladder, ammonium/ammonia, the two-step carbonate ladder, and
acetic acid's own dissociation) and the same `Ecoli`/`AceticAcid` species
declarations as
[usecase 03 §2](03_grow_ecoli_on_acetic_acid.ipynb#2--Declare-the-chemistry)
— nothing about the acid-base network changes when the reactor becomes
continuous instead of batch.

In [ ]:
ACETIC_ACID = Species(
    id="AceticAcid", atoms={"C": 2, "H": 4, "O": 2}, charge=0, MW=60.052,
)
ACETATE_MINUS = Species(
    id="Acetate-", atoms={"C": 2, "H": 3, "O": 2}, charge=-1, MW=59.044,
)
ECOLI = Species(
    id="Ecoli", atoms={"C": 1, "H": 1.8, "O": 0.5, "N": 0.2}, charge=0,
    MW=24.626,
)

water = EquilibriumReaction(
    stoichiometry=[_e(H2O, -1), _e(H_plus, +1), _e(OH_minus, +1)],
    log_K=-14.0, label="water",
)
p1 = EquilibriumReaction(
    stoichiometry=[_e(H3PO4, -1), _e(H2PO4_minus, +1), _e(H_plus, +1)],
    log_K=-2.15, total_id="H3PO4", label="p1",
)
p2 = EquilibriumReaction(
    stoichiometry=[_e(H2PO4_minus, -1), _e(HPO4_2minus, +1), _e(H_plus, +1)],
    log_K=-7.20, total_id="H3PO4", label="p2",
)
p3 = EquilibriumReaction(
    stoichiometry=[_e(HPO4_2minus, -1), _e(PO4_3minus, +1), _e(H_plus, +1)],
    log_K=-12.35, total_id="H3PO4", label="p3",
)
nh4 = EquilibriumReaction(
    stoichiometry=[_e(NH4_plus, -1), _e(NH3, +1), _e(H_plus, +1)],
    log_K=-9.25, total_id="NH3", label="nh4",
)
co2_first = EquilibriumReaction(
    stoichiometry=[_e(CO2, -1), _e(H2O, -1), _e(HCO3_minus, +1), _e(H_plus, +1)],
    log_K=-6.35, total_id="CO2", label="co2_first",
)
co2_second = EquilibriumReaction(
    stoichiometry=[_e(HCO3_minus, -1), _e(CO3_2minus, +1), _e(H_plus, +1)],
    log_K=-10.33, total_id="CO2", label="co2_second",
)
acetate_eq = EquilibriumReaction(
    stoichiometry=[_e(ACETIC_ACID, -1), _e(ACETATE_MINUS, +1), _e(H_plus, +1)],
    log_K=-4.756, total_id="AceticAcid", label="acetate_eq",
)

print("8 equilibrium reactions declared (unchanged from usecase 03).")


## 2  Monod growth kinetics for continuous operation

$\mu_{max}$, the yield $Y_{X/S}$, and $K_{O_2}$ carry over from usecase 03
unchanged; $K_s$ is set differently here. A chemostat's steady-state
substrate concentration is

$$S_{ss} = \frac{K_s \, D}{\mu_{max} - D}$$

which, at usecase 03's $K_s = 5$ mg/L (a high-affinity estimate appropriate
for a *batch*, where substrate only approaches zero briefly at the end of
a run), sits at sub-mg/L levels across almost the entire viable
dilution-rate range in a *continuous* culture — a numerically stiff regime
for the explicit-Euler reaction sub-stepping this framework uses. This
notebook instead uses $K_s = 0.5$ g/L (500 mg/L): still within the
order-of-magnitude spread usecase 03's own literature review noted for
organic-acid uptake kinetics (Kovarova-Kovar & Egli 1998), and a
substrate-affinity choice rather than a growth-rate or yield change. It
keeps $S_{ss}$ a comfortable double-digit-percent fraction of the feed
concentration everywhere except right at washout, so the dilution-rate
sweep in Section 6 runs at a normal, non-stiff step size.

In [ ]:
mu_max_per_h = 0.30    # h^-1              -- unchanged from usecase 03
Ks_gL        = 0.5     # g acetate / L     -- see Section 2 above
Yxs          = 0.36    # g biomass / g acetate -- unchanged
Ko2_gL       = 0.2e-3  # g O2 / L          -- unchanged

growth = ReactionBuilder.monod_aerobic_growth(
    substrate=ACETIC_ACID, biomass=ECOLI,
    mu_max_per_h=mu_max_per_h, Ks_gL=Ks_gL, yield_gX_gS=Yxs, Ko2_gL=Ko2_gL,
    balance="CHNO", label="growth_on_AceticAcid",
)
system = ReactionSystem(
    [water, p1, p2, p3, nh4, co2_first, co2_second, acetate_eq, growth],
    label="ecoli_on_acetate",
)

print(f"mu_max = {mu_max_per_h} /h, Ks = {Ks_gL*1e3:.0f} mg/L, "
      f"Yxs = {Yxs} g/g, Ko2 = {Ko2_gL*1e3} mg/L")


## 3  Chemostat theory: a closed-form sanity check

This is standard chemostat theory (Monod 1950; see e.g. Bailey & Ollis,
*Biochemical Engineering Fundamentals*, ch. 7, or Blanch & Clark,
*Biochemical Engineering*, ch. 6) — restated here to pick a sensible
dilution-rate range for Section 6's sweep and to sanity-check the
mechanistic simulation in Section 5. It's a reference line on the plots
that follow, not a second model this notebook tries to validate in its
own right.

At steady state, growth exactly balances washout: $\mu(S_{ss}) = D$.
Substituting the Monod form $\mu(S) = \mu_{max}\,S/(K_s+S)$ and solving
for $S$ gives the **substrate-only** estimate

$$S_{ss}(D) = \frac{K_s \, D}{\mu_{max} - D} \qquad (D < \mu_{max})$$

and the biomass estimate follows from the yield alone (the same
`ReactionBuilder`-derived stoichiometry usecase 03 §6 validated by mass
balance):

$$X_{ss}(D) = Y_{X/S}\,\bigl(S_{feed} - S_{ss}(D)\bigr)$$

This closed form assumes dissolved O₂ never limits growth (the O₂ Monod
term $O_2/(K_{O_2}+O_2)$ from Section 2 is implicitly treated as exactly
1, i.e. DO always sits at air saturation) — an assumption Section 6's own
DO panel shows holds closely only for part of the swept range, since the
vessel's aeration capacity is finite. Two quantities from this
substrate-only estimate are still useful as *reference lines*:

- **Washout.** As $D \to \mu_{max}$, $S_{ss} \to \infty$ in the formula
  above, which is unphysical — what actually happens is $X_{ss}$ hits zero
  *before* that, at $S_{ss} = S_{feed}$ (all the feed passes through
  unconsumed). Solving $S_{ss}(D_{crit}) = S_{feed}$ gives the **washout
  dilution rate**
  $$D_{crit} = \mu_{max}\,\frac{S_{feed}}{K_s + S_{feed}}$$
  Because this closed form ignores O₂ limitation, it's an *upper bound* on
  where the mechanistic model actually washes out — Section 6's simulated
  values fall away from it well before reaching $D_{crit}$ itself.
- **Volumetric productivity.** The reactor's output of biomass per unit
  time and volume is $P(D) = D\,X_{ss}(D)$ — zero at $D=0$ (no flow, no
  output) and zero at $D_{crit}$ (washout, no biomass), so it has an
  interior maximum. Differentiating and setting $dP/dD=0$ gives the
  **productivity-optimal dilution rate**
  $$D_{opt} = \mu_{max}\left(1 - \sqrt{\frac{K_s}{K_s + S_{feed}}}\right)$$
  Section 6 checks how close the mechanistic model's *own* simulated
  productivity peak sits to this estimate.

In [ ]:
C_AcOH_feed_gL = 4.0   # g/L acetic acid in the feed (same loading as usecase 03)

def S_ss_analytic(D, Ks=Ks_gL, mu_max=mu_max_per_h):
    return Ks * D / (mu_max - D)

def X_ss_analytic(D, Sfeed=C_AcOH_feed_gL, Ks=Ks_gL, mu_max=mu_max_per_h, Y=Yxs):
    return Y * (Sfeed - S_ss_analytic(D, Ks, mu_max))

D_crit = mu_max_per_h * C_AcOH_feed_gL / (Ks_gL + C_AcOH_feed_gL)
D_opt  = mu_max_per_h * (1.0 - np.sqrt(Ks_gL / (Ks_gL + C_AcOH_feed_gL)))

print(f"S_feed              = {C_AcOH_feed_gL} g/L acetic acid")
print(f"D_crit (substrate-only washout)    = {D_crit:.4f} /h   (tau = {1/D_crit:.2f} h)")
print(f"D_opt (substrate-only max productivity) = {D_opt:.4f} /h   (tau = {1/D_opt:.2f} h)")
print(f"At D_opt: S_ss = {S_ss_analytic(D_opt):.4f} g/L, "
      f"X_ss = {X_ss_analytic(D_opt):.4f} g/L, "
      f"productivity = {D_opt*X_ss_analytic(D_opt):.4f} g/L/h")


## 4  Assemble the chemostat and run one dilution rate to steady state

Same 2 L vessel, headspace, and aeration pattern as usecase 03 (`GasFeed` +
`PressureReliefVent`, kinetic O₂/CO₂/N₂ transfer) — none of
that changes. `GasFeed`'s aeration rate (1 vvm — matching the airflow
Leone et al. 2015 used per litre of working volume, see reference below)
is fixed, and so is gas-liquid transfer: $k_La$ = 90 /h for O₂, CO₂, and N₂ throughout this
notebook, within the range reported for bench-scale *E. coli* STRs under
mechanical agitation/sparging — roughly 70-1000 /h across reported
operating points (~72 /h at 200 rpm/1.5 vvm in a 5 L fermenter, up to
~980 /h at 1500 rpm/18 L min⁻¹ in a 12.8 L bioreactor; see references
below) — and toward the moderate end of it, comfortably inside the regime
where Section 6's DO panel shows aeration capacity actually starts to
matter.

What's new relative to usecase 03 is the liquid boundary pair that makes
this a chemostat instead of a bottle: `LiquidFeed` delivers sterile medium
(phosphate, ammonium, and acetic acid at their feed concentrations — *no*
biomass, since the feed stream is sterile) at rate $Q$, and `LiquidDrain`
removes broth (every dissolved species, biomass included) at that same
$Q$, so $V_{liq}$ stays fixed by construction. `build_cv(D_per_h)` builds
one fresh chemostat at a given dilution rate; it's called once below for
$D=0.15$/h and again in Section 6 across the full sweep.

*References:* Leone et al. (2015, *Microb. Cell Fact.* 14:106) — the same
paper cited for $Y_{X/S}$ in Section 2 and the feed loading below — report
an airflow of 60 L/h into a 1 L working volume (1 vvm, matching this
vessel's `vvm_min`). For the $k_La$ range: BioProcess International,
"Lessons in Bioreactor Scale-Up, Part 4" (5 L fermenter, 200 rpm/1.5 vvm
example); Fan et al., *J. Chem. Pharm. Res.*, 2014, 6(7):1810-1817 (12.8 L
bioreactor, recombinant *E. coli*, 1500 rpm/18 L min⁻¹ example).

In [ ]:
V_total_L = 2.0
headspace_frac = 0.20
V_liq = V_total_L * (1.0 - headspace_frac)
V_gas = V_total_L * headspace_frac
T_K = 310.15   # 37 C, same as usecase 03

CT_P = 0.022     # mol/L KH2PO4 -> mol/L total phosphate, mol/L K+  (usecase 01's recipe)
CT_N = 0.0187    # mol/L NH4Cl  -> mol/L total ammoniacal N, mol/L Cl-
C_AcOH_feed = C_AcOH_feed_gL / ACETIC_ACID.MW

kLa = 90.0   # /h for O2, CO2, and N2 -- see Section 4 above

def kH_mol_L_atm(H_ref, dlnH, T_K, T_ref=298.15):
    H_ref_mol_L_atm = (H_ref / 1000.0) * 101325.0
    return H_ref_mol_L_atm * math.exp(dlnH * (1.0 / T_K - 1.0 / T_ref))

DO_sat_mgL = kH_mol_L_atm(1.3e-5, 1500.0, T_K) * 0.2095 * 32.00 * 1e3   # air-sat. DO, 37 C
Ko2_mgL = Ko2_gL * 1e3

def make_transfer_models():
    return {
        "O2": KineticTransferModel(
            partition_model=HenryEquilibrium(H_ref=1.3e-5, dlnH=1500.0), k_transfer=kLa,
        ),
        "CO2": KineticTransferModel(
            partition_model=HenryEquilibrium(H_ref=3.4e-4, dlnH=2400.0), k_transfer=kLa,
        ),
        "N2": KineticTransferModel(
            partition_model=HenryEquilibrium(H_ref=6.4e-6, dlnH=1300.0), k_transfer=kLa,
        ),
    }

def build_cv(D_per_h, X0_gL=0.05):
    Q_L_per_h = D_per_h * V_liq
    n_total_gas = (1.0 * V_gas) / (R_L_ATM_MOL_K * T_K)
    gas_phase = GasPhase(
        n_mol={
            "O2": n_total_gas * 0.2095, "N2": n_total_gas * 0.7901,
            "CO2": n_total_gas * 0.0004,
        },
        V_L=V_gas, T_K=T_K,
    )
    X0 = X0_gL / ECOLI.MW
    liquid_phase = LiquidPhase(
        n_mol={
            "H3PO4": CT_P * V_liq, "NH3": CT_N * V_liq,
            "AceticAcid": C_AcOH_feed * V_liq, "Ecoli": X0 * V_liq,
            "K+": CT_P * V_liq, "Cl-": CT_N * V_liq,
            "O2": 0.0, "CO2": 0.0, "N2": 0.0,
        },
        V_L=V_liq, T_K=T_K,
    )
    boundaries = [
        GasFeed(vvm_min=1.0, y={"O2": 0.2095, "N2": 0.7901, "CO2": 0.0004},
                P_inlet_atm=1.0, phase_key="gas", liquid_phase_key="liquid",
                label="gas_feed"),
        PressureReliefVent(P_set_atm=1.0, mode="instant"),
        LiquidFeed(
            Q_L_per_h=Q_L_per_h,
            feed_conc_mol_L={
                "H3PO4": CT_P, "NH3": CT_N, "AceticAcid": C_AcOH_feed,
                "K+": CT_P, "Cl-": CT_N,
            },
            label="sterile_feed",
        ),
        LiquidDrain(Q_L_per_h=Q_L_per_h, label="broth_drain"),
    ]
    return ControlVolume(
        phases={"gas": gas_phase, "liquid": liquid_phase},
        transfer_models=make_transfer_models(), boundaries=boundaries,
        reaction_system=system, label=f"cstr_D{D_per_h:.3f}",
    )

def run_to_steady_state(D_per_h, n_res=15.0, dt_h=0.01):
    """Run build_cv(D_per_h) for n_res multiples of the system's own
    relaxation time 1/(mu_max-D) -- *not* the residence time 1/D, which
    underestimates how long convergence actually takes near D_opt/washout
    (Section 6 discusses this)."""
    cv = build_cv(D_per_h)
    tau_relax_h = 1.0 / (mu_max_per_h - D_per_h)
    tau_sim_h = n_res * tau_relax_h
    n_steps = int(tau_sim_h / dt_h)
    sim = Simulation(cvs={"main": cv}, label=f"cstr_D{D_per_h:.3f}")
    result = sim.run(tau_h=tau_sim_h, n_steps=n_steps)
    return result, tau_sim_h

D_demo = 0.15   # the dilution rate this section runs and checks in detail
result_demo, tau_sim_demo = run_to_steady_state(D_demo)
print(f"D = {D_demo}/h  (kLa={kLa:.0f}/h, tau_residence = {1/D_demo:.2f} h), "
      f"ran {tau_sim_demo:.1f} h simulated in {result_demo.runtime_s:.1f} s wall-clock.")
print(f"DO_sat = {DO_sat_mgL:.2f} mg/L (air saturation, 37 C), K_O2 = {Ko2_mgL:.2f} mg/L")


## 5  Approach to steady state, vs. the substrate-only estimate

Biomass and substrate both start at their inoculum/feed values and relax
toward steady state — plotted against the substrate-only closed form from
Section 3 (gray, dashed) for this same $D=0.15$/h, as a sanity check that
the CV's feed/drain/reaction/transfer machinery, run through a full
`Simulation`, reproduces basic chemostat behaviour. The two agree closely
here (well under 1%) — Section 6's sweep shows that agreement eroding
substantially above $D\approx0.18$-$0.20$/h, where the vessel's finite
aeration capacity starts to bind.

In [ ]:
t = result_demo.t_h
X = result_demo.liquid_mol["main"]["Ecoli"] * ECOLI.MW / V_liq
S = result_demo.liquid_mol["main"]["AceticAcid"] * ACETIC_ACID.MW / V_liq
pH = result_demo.pH["main"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].plot(t, X, color="tab:green")
axes[0].axhline(X_ss_analytic(D_demo), color="gray", ls="--", lw=1,
                label=f"substrate-only X_ss = {X_ss_analytic(D_demo):.3f} g/L")
axes[0].set_xlabel("time (h)")
axes[0].set_ylabel("biomass (g/L)")
axes[0].set_title(f"Biomass approaching steady state, D={D_demo}/h")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(t, S, color="tab:blue")
axes[1].axhline(S_ss_analytic(D_demo), color="gray", ls="--", lw=1,
                label=f"substrate-only S_ss = {S_ss_analytic(D_demo):.3f} g/L")
axes[1].set_xlabel("time (h)")
axes[1].set_ylabel("acetic acid (g/L)")
axes[1].set_title(f"Substrate approaching steady state, D={D_demo}/h")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"X_ss: simulated = {X[-1]:.4f} g/L  |  substrate-only estimate = "
      f"{X_ss_analytic(D_demo):.4f} g/L  |  gap = "
      f"{100*abs(X[-1]-X_ss_analytic(D_demo))/X_ss_analytic(D_demo):.1f}%")
print(f"S_ss: simulated = {S[-1]:.4f} g/L  |  substrate-only estimate = "
      f"{S_ss_analytic(D_demo):.4f} g/L  |  gap = "
      f"{100*abs(S[-1]-S_ss_analytic(D_demo))/S_ss_analytic(D_demo):.1f}%")
print(f"pH at steady state: {pH[-1]:.2f}")


That last line is worth pausing on: pH settles to a low, *uncontrolled*
value here, well below usecase 01's phosphate-only baseline (pH ≈ 4.7).
Two things conspire, both structural to running this culture
continuously rather than in a batch: (1) growth keeps drawing down the
ammoniacal-nitrogen pool for biomass synthesis while phosphate (which
nothing consumes) stays at its feed concentration, so the medium's
buffer composition shifts at steady state in a way it never fully does
in a batch; and (2) aerobic growth on an organic acid continuously
produces CO₂, adding a steady carbonic-acid load that a batch run only
accumulates transiently before consumption stops. Usecase 03 could let pH
drift freely because a batch run ends; a chemostat runs indefinitely at
whatever pH this settles to, which is precisely why real continuous
cultures on acid substrates are normally run under active pH control —
`demos/builder/cstr_fermenter.py`'s `PHController` (dosing NaOH to a
setpoint) is the closed-loop fix, left out here so this notebook can stay
focused on the dilution-rate/productivity relationship itself.

## 6  Dilution-rate sweep and the summary figure

This section sweeps $D$ from a low dilution rate up toward washout,
running the *same* single mechanistic model from Sections 4-5 at each
point and reading its steady state directly: biomass, substrate,
dissolved O₂, and pH all come out of the one `ControlVolume`/`Simulation`
run, with no separate O₂- or pH-specific model needed to get at any of
them.

Points closer to washout take longer to reach steady state — critical
slowing down, a well-documented chemostat phenomenon (perturbations relax
more slowly the closer the operating point sits to the washout boundary;
see e.g. Bailey & Ollis ch. 7) — so `run_to_steady_state`'s `n_res` is
bumped for the last, slowest point. The sweep stops at $D=0.24$/h:
pushing closer to Section 3's substrate-only $D_{crit}$ (0.267/h) runs
into that slowing-down directly, to the point where confirming
convergence would cost far more simulated time than this sweep is worth.
That the mechanistic model is already declining well before $D_{crit}$ is
itself the point (Section 3): the substrate-only closed form assumes O₂
is never limiting, and this vessel's finite aeration capacity means the
real washout point sits below 0.267/h, not at it.

In [ ]:
D_sweep = [0.02, 0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.22, 0.23, 0.24]

X_ss, S_ss, DO_ss, pH_ss = {}, {}, {}, {}
X_ss[D_demo], S_ss[D_demo] = X[-1], S[-1]
DO_ss[D_demo] = result_demo.liquid_mol["main"]["O2"][-1] * 32.00 * 1e3 / V_liq
pH_ss[D_demo] = pH[-1]

for D in D_sweep:
    if D == D_demo:
        continue
    n_res = 40.0 if D >= 0.24 else 15.0
    result_i, tau_i = run_to_steady_state(D, n_res=n_res)
    X_ss[D] = result_i.liquid_mol["main"]["Ecoli"][-1] * ECOLI.MW / V_liq
    S_ss[D] = result_i.liquid_mol["main"]["AceticAcid"][-1] * ACETIC_ACID.MW / V_liq
    DO_ss[D] = result_i.liquid_mol["main"]["O2"][-1] * 32.00 * 1e3 / V_liq
    pH_ss[D] = result_i.pH["main"][-1]
    print(f"D={D:.3f}/h  n_res={n_res:.0f}  tau_sim={tau_i:6.1f}h  "
          f"X_ss={X_ss[D]:.4f} g/L  S_ss={S_ss[D]:.4f} g/L  "
          f"DO_ss={DO_ss[D]:.3f} mg/L  pH_ss={pH_ss[D]:.2f}")

D_all = sorted(X_ss)
P_ss = {D: D * X_ss[D] for D in D_all}
D_opt_sim = max(D_all, key=lambda D: P_ss[D])

print(f"\nSwept D = {D_all} /h")
print(f"Simulated productivity peaks at D={D_opt_sim}/h (P={P_ss[D_opt_sim]:.4f} g/L/h), "
      f"vs. the substrate-only estimate D_opt={D_opt:.4f}/h.")


### 6.1  Runtime benchmark

For the manuscript runtime table, this cell runs three representative
dilution rates once each through the same `run_to_steady_state` helper used
above. The reported time is `SimulationResult.runtime_s × 1000`, so each
row is the wall-clock time for one full CSTR simulation at that dilution
rate.

In [ ]:
D_runtime = [0.01, 0.10, 0.20, 0.219]
N_RUNTIME_REPS = 1

print(f"{'Dilution rate':>14}{'Run Time, ms':>16}")
cstr_runtime_rows = []
for D in D_runtime:
    times_ms = []
    for _ in range(N_RUNTIME_REPS):
        result_i, _tau_i = run_to_steady_state(D)
        times_ms.append(result_i.runtime_s * 1e3)
    runtime_ms = float(np.mean(times_ms))
    cstr_runtime_rows.append((D, runtime_ms))
    print(f"{D:>14.2f}{runtime_ms:>16.1f}")


### 6.2  Publication figure

Single column, three linked panels sharing the $D$ axis. The middle panel
is split into two tightly-coupled sub-panels (DO above, pH below) rather
than overlaid on one dual-axis plot: dissolved O₂ and pH sit on
incompatible scales, and a shared-axis overlay would misrepresent both.

(a) steady-state substrate and biomass vs. $D$, against the substrate-only
estimate from Section 3; (b)/(c) steady-state dissolved O₂ and pH vs. $D$
— (b) also marks air-saturation $DO_{sat}$ and the O₂ half-saturation
constant $K_{O_2}$ for scale; (d) volumetric productivity
$P(D)=D\,X_{ss}(D)$ vs. $D$, marking both the substrate-only $D_{opt}$
and where the simulated productivity actually peaks. The gray dotted line
in each panel is Section 3's substrate-only $D_{crit}$ — a reference
ceiling the mechanistic model falls short of, not a value it's expected
to reach.

In [ ]:
D_arr = np.array(D_all)
X_arr = np.array([X_ss[D] for D in D_all])
S_arr = np.array([S_ss[D] for D in D_all])
DO_arr = np.array([DO_ss[D] for D in D_all])
pH_arr = np.array([pH_ss[D] for D in D_all])
P_arr = np.array([P_ss[D] for D in D_all])

D_theory = np.linspace(1e-4, D_crit * 0.999, 400)

# Fixed categorical colors (identity, not rank) -- consistent across panels.
C_BIOMASS, C_SUBSTRATE, C_DO, C_PH, C_PROD = (
    "#2a78d6", "#eb6834", "#1baf7a", "#008300", "#4a3aa7",
)
C_MUTED, C_GRID = "#898781", "#e1e0d9"

PUB_FIG_WIDTH_IN = 3.5   # single-column journal width, same convention as usecase 01
PUB_DPI = 600
PUB_STYLE = {
    "font.size": 8, "axes.labelsize": 9, "axes.titlesize": 9,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 6.5,
    "axes.edgecolor": C_MUTED, "axes.labelcolor": "#0b0b0b",
    "xtick.color": C_MUTED, "ytick.color": C_MUTED,
    "axes.grid": True, "grid.color": C_GRID, "grid.linewidth": 0.6,
    "axes.linewidth": 0.6,
}

FIG_DIR = _find_repo() / "demos" / "usecases" / "figures"
FIG_DIR.mkdir(exist_ok=True)

def _mark_D(ax, label=False):
    ax.axvline(D_crit, color=C_MUTED, ls=":", lw=1,
               label=f"$D_{{crit}}$ (substrate-only) = {D_crit:.3f} /h" if label else None)
    ax.axvline(D_opt, color=C_MUTED, ls="--", lw=1,
               label=f"$D_{{opt}}$ (substrate-only) = {D_opt:.3f} /h" if label else None)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

with plt.rc_context(PUB_STYLE):
    fig = plt.figure(figsize=(PUB_FIG_WIDTH_IN, 9.5))
    outer = fig.add_gridspec(3, 1, height_ratios=[1.15, 1.7, 1.15], hspace=0.42)
    ax_a = fig.add_subplot(outer[0])
    inner_bc = outer[1].subgridspec(2, 1, hspace=0.40)
    ax_b = fig.add_subplot(inner_bc[0], sharex=ax_a)
    ax_c = fig.add_subplot(inner_bc[1], sharex=ax_a)
    ax_d = fig.add_subplot(outer[2], sharex=ax_a)

    # (a) substrate & biomass
    ax_a.plot(D_theory, X_ss_analytic(D_theory), color=C_BIOMASS, ls=":", lw=1, alpha=0.6)
    ax_a.plot(D_theory, S_ss_analytic(D_theory), color=C_SUBSTRATE, ls=":", lw=1, alpha=0.6,
              label="substrate-only estimate")
    ax_a.plot(D_arr, X_arr, color=C_BIOMASS, marker="o", ms=3.5, lw=1.4, label="biomass $X_{ss}$")
    ax_a.plot(D_arr, S_arr, color=C_SUBSTRATE, marker="o", ms=3.5, lw=1.4, label="substrate $S_{ss}$")
    _mark_D(ax_a, label=True)
    ax_a.set_ylabel("conc. (g/L)")
    ax_a.set_title("(a)  Steady-state substrate & biomass", loc="left")
    ax_a.legend(frameon=False, loc="upper left")
    plt.setp(ax_a.get_xticklabels(), visible=False)

    # (b) dissolved O2
    ax_b.plot(D_arr, DO_arr, color=C_DO, marker="o", ms=3.5, lw=1.4)
    ax_b.axhline(DO_sat_mgL, color=C_MUTED, ls="--", lw=0.8)
    ax_b.axhline(Ko2_mgL, color=C_MUTED, ls=":", lw=0.8)
    ax_b.text(D_arr[-1], DO_sat_mgL, "$DO_{sat}$ ", color=C_MUTED, fontsize=6.5,
              va="bottom", ha="right")
    ax_b.text(D_arr[-1], Ko2_mgL, "$K_{O_2}$ ", color=C_MUTED, fontsize=6.5,
              va="bottom", ha="right")
    ax_b.set_ylim(-0.5, DO_sat_mgL * 1.15)
    _mark_D(ax_b)
    ax_b.set_ylabel("DO (mg/L)")
    ax_b.set_title("(b)  Steady-state dissolved O$_2$", loc="left")
    plt.setp(ax_b.get_xticklabels(), visible=False)

    # (c) pH
    ax_c.plot(D_arr, pH_arr, color=C_PH, marker="o", ms=3.5, lw=1.4)
    _mark_D(ax_c)
    ax_c.set_ylabel("pH")
    ax_c.set_title("(c)  Steady-state pH", loc="left")
    plt.setp(ax_c.get_xticklabels(), visible=False)

    # (d) productivity
    ax_d.plot(D_theory, D_theory * X_ss_analytic(D_theory), color=C_MUTED, ls=":", lw=1,
              alpha=0.6, label="substrate-only estimate")
    ax_d.plot(D_arr, P_arr, color=C_PROD, marker="o", ms=3.5, lw=1.4, label="simulated $P(D)$")
    ax_d.scatter([D_opt_sim], [P_ss[D_opt_sim]], color=C_PROD, edgecolor="#0b0b0b", s=45,
                 zorder=5, label=f"simulated peak, D={D_opt_sim:.2f}/h")
    _mark_D(ax_d)
    ax_d.set_xlabel("dilution rate $D$ (1/h)")
    ax_d.set_ylabel("productivity (g/L/h)")
    ax_d.set_title("(d)  Volumetric productivity", loc="left")
    ax_d.legend(frameon=False, loc="upper left")

    for ax in (ax_a, ax_b, ax_c, ax_d):
        ax.set_xlim(0, D_crit * 1.02)

    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"05_cstr_dilution_rate_sweep.{ext}",
                    dpi=PUB_DPI, bbox_inches="tight")
    plt.show()

print(f"Saved publication figure to {FIG_DIR / '05_cstr_dilution_rate_sweep.png'}")


## Takeaways

- **The dilution rate is the whole story for a fixed reactor.** Given
  fixed $S_{feed}$, $\mu_{max}$, $K_s$, $Y_{X/S}$, and $k_La$, every
  steady-state quantity — substrate residual, biomass concentration,
  dissolved O₂, pH, volumetric productivity — is a function of $D$ alone.
  Turning that one dial moves the operating point along the curves in
  Section 6.1's figure.
- **A single mechanistic model gets all four quantities at once.** Because
  the `ControlVolume`/`Simulation` run tracks O₂ transfer and acid-base
  speciation directly, there's no need for a separate closed-form O₂ model
  to get at DO, or a separate calculation to get at pH — Section 6's sweep
  reads all of it off the same run that also gives $S_{ss}$ and $X_{ss}$.
- **"DO looks healthy" is not the same as "DO isn't affecting the
  answer."** Dissolved O₂ never crashes anywhere in this sweep — it stays
  within a few mg/L of air saturation throughout (comfortably non-limiting
  on any DO probe reading) — yet it's still responsible for a real,
  measurable shift in $S_{ss}$/$X_{ss}$/productivity relative to the
  substrate-only estimate, growing from under 1% at $D=0.15$/h to over 10%
  by $D=0.23$/h (Section 5, Section 6.1's panel a).
- **Productivity is maximised strictly below washout.** The substrate-only
  estimate puts that optimum at
  $D_{opt} = \mu_{max}(1-\sqrt{K_s/(K_s+S_{feed})})$ — not at $D_{crit}$
  itself, and the gap between them is the margin a real operator has
  before a disturbance tips the culture into washout. Section 6.1 shows
  where the mechanistic model's own simulated productivity actually peaks,
  which need not sit at exactly the same $D$ once O₂ limitation is
  accounted for.
- **Convergence time is not symmetric around washout.** Approaching the
  mechanistic model's own washout point from below is markedly slower —
  critical slowing down, a well-documented chemostat phenomenon — which is
  exactly why this notebook's sweep stops at $D=0.24$/h rather than
  pushing all the way to Section 3's substrate-only $D_{crit}$ (Section 6).

## Where to go next

- **Vary $k_La$ yourself.** `build_cv()` and `run_to_steady_state()` take
  no aeration argument in this notebook because a single condition was
  kept for simplicity — reintroducing `kLa` as a parameter and re-running
  Section 6's sweep at a second value (e.g. a more vigorously sparged
  vessel) shows directly how much of the gap from the substrate-only
  estimate is an aeration artifact rather than a fixed property of this
  organism/substrate pair.
- **Closed-loop pH (and DO) control for this exact chemostat** —
  `demos/builder/cstr_fermenter.py`'s `PHController`/`DOAgitationController`
  cascade, addressing the low-pH note in Section 5.
- **Back to batch, for comparison** —
  [usecase 03](03_grow_ecoli_on_acetic_acid.ipynb) runs the identical
  organism/substrate/kinetics without feed or drain at all.
- **`FermenterBuilder`, the fluent alternative to hand-assembling the
  `ControlVolume` in Section 4** —
  `models/vlmodels/fermenter/config/builder.py`, used directly by
  `demos/builder/cstr_fermenter.py`.